# Colorado Wildfire Fire/Non-Fire Prediction
## Google Earth Engine · AlphaEarth Embeddings · XGBoost · Random Forest

**Study area:** Colorado  
**Fire perimeters:** USFS MTBS (`USFS/GTAC/MTBS/burned_area_boundaries/v1`)  
**Predictors:** AlphaEarth satellite embeddings · TerraClimate climate · SRTM topography  
**Models:** XGBoost classifier · Random Forest classifier · XGBoost area regressor (LOO-CV)  
**App:** Streamlit Digital Twin with live GEE embedding queries

---

## Table of Contents
1. [Setup](#1-setup)
2. [Explore MTBS Schema](#2-explore-mtbs-schema)
3. [Data Collection — Polygon Extraction](#3-data-collection--polygon-extraction)
4. [Data Collection — Centroid Buffer Extraction](#4-data-collection--centroid-buffer-extraction)
5. [Training Dataset — Multi-year Fire/Non-fire](#5-training-dataset--multi-year-firenon-fire)
6. [Training Dataset — 2020 Fire/Non-fire (Clean Non-fire)](#6-training-dataset--2020-firenon-fire-clean-non-fire)
7. [Export Shapefiles](#7-export-shapefiles)
8. [Final Pre-fire Training Dataset (Recommended)](#8-final-pre-fire-training-dataset-recommended)
9. [XGBoost Classifier](#9-xgboost-classifier)
10. [Random Forest Classifier](#10-random-forest-classifier)
11. [XGBoost Area Regression (LOO-CV)](#11-xgboost-area-regression-loo-cv)
12. [Streamlit Digital Twin App](#12-streamlit-digital-twin-app)

## 1. Setup

Authenticate with Google Earth Engine and install required packages.

In [ ]:
import ee

ee.Authenticate()
ee.Initialize()

In [ ]:
pip install xgboost geemap geopandas

## 2. Explore MTBS Schema

Inspect property names and counts for the MTBS burned area boundary collection
filtered to Colorado. The key date field is `Ig_Date` (ignition date).

In [ ]:
import ee
ee.Initialize()

# Colorado boundary
states   = ee.FeatureCollection("TIGER/2018/States")
colorado = states.filter(ee.Filter.eq("NAME", "Colorado")).geometry()

# MTBS collection
mtbs  = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
first = mtbs.first()

print("Property names:")
print(first.propertyNames().getInfo())

print("\nFirst feature:")
print(first.toDictionary().getInfo())

print("\nTotal MTBS fires in Colorado:")
print(mtbs.filterBounds(colorado).size().getInfo())

### Confirm year field

Probe candidate year-field names to confirm that fire year must be
derived from `Ig_Date` rather than read directly.

In [ ]:
possible_fields = ["Ig_Year", "Fire_Year", "YEAR", "Year", "FIRE_YEAR", "Incid_Year"]

for field in possible_fields:
    try:
        n = (mtbs.filterBounds(colorado)
                  .filter(ee.Filter.eq(field, 2020))
                  .size().getInfo())
        print(f"{field}: 2020 count = {n}")
    except Exception:
        print(f"{field}: field not found")

## 3. Data Collection — Polygon Extraction

Extract mean AlphaEarth embeddings (before / after / delta) and GRIDMET
fire-weather variables over full MTBS fire polygons for 2020.

**Output:** `CO_2020_MTBS_20_AEF_GRIDMET_training_table.csv`

In [ ]:
import ee
import geemap
import pandas as pd

ee.Initialize()

# ── Parameters ────────────────────────────────────────────────────────────────
FIRE_YEAR   = 2020
BEFORE_YEAR = 2019
AFTER_YEAR  = 2021
N_FIRES     = 20
FIRE_DATE_FIELD = "Ig_Date"
OUT_CSV = f"CO_{FIRE_YEAR}_MTBS_{N_FIRES}_AEF_GRIDMET_training_table.csv"

# ── Study area ────────────────────────────────────────────────────────────────
states   = ee.FeatureCollection("TIGER/2018/States")
colorado = states.filter(ee.Filter.eq("NAME", "Colorado")).geometry()

# ── MTBS: add fire year derived from Ig_Date ──────────────────────────────────
mtbs = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")

def add_fire_year(f):
    d = ee.Date(f.get(FIRE_DATE_FIELD))
    return f.set({"fire_date": d.format("YYYY-MM-dd"),
                  "fire_year_from_date": d.get("year")})

fires = (mtbs.map(add_fire_year)
             .filterBounds(colorado)
             .filter(ee.Filter.eq("fire_year_from_date", FIRE_YEAR))
             .limit(N_FIRES))

print("Selected fires:", fires.size().getInfo())

# ── Add area and centroid metadata ────────────────────────────────────────────
def add_area(f):
    area_ha  = f.geometry().area(maxError=30).divide(10000)
    centroid = f.geometry().centroid(maxError=30).coordinates()
    return f.set({"event_id":     ee.String("fire_").cat(ee.String(f.id())),
                  "fire_year":    FIRE_YEAR,
                  "burn_area_ha": area_ha,
                  "lon":          centroid.get(0),
                  "lat":          centroid.get(1)})

fires = fires.map(add_area)

# ── AlphaEarth embeddings: before / after / delta ─────────────────────────────
emb = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

emb_before = (emb.filterDate(f"{BEFORE_YEAR}-01-01", f"{BEFORE_YEAR}-12-31")
                  .first().resample("bilinear").reproject(crs="EPSG:4326", scale=30))
emb_after  = (emb.filterDate(f"{AFTER_YEAR}-01-01", f"{AFTER_YEAR}-12-31")
                  .first().resample("bilinear").reproject(crs="EPSG:4326", scale=30))

bands  = emb_before.bandNames()
before = emb_before.rename(bands.map(lambda b: ee.String("before_").cat(b)))
after  = emb_after.rename( bands.map(lambda b: ee.String("after_").cat(b)))
delta  = emb_after.subtract(emb_before).rename(
             bands.map(lambda b: ee.String("delta_").cat(b)))

embedding_stack = before.addBands(after).addBands(delta).reproject(crs="EPSG:4326", scale=30)

# ── GRIDMET fire-weather: 7-day window around ignition ───────────────────────
gridmet = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")

def add_fire_weather(f):
    fire_date = ee.Date(f.get(FIRE_DATE_FIELD))
    window    = gridmet.filterDate(fire_date.advance(-3, "day"), fire_date.advance(4, "day"))

    def zonal(img, band):
        return img.reduceRegion(reducer=ee.Reducer.mean(), geometry=f.geometry(),
                                scale=4000, crs="EPSG:4326", maxPixels=1e9,
                                bestEffort=True, tileScale=4).get(band)

    return f.set({
        "vpd_fire_day": zonal(gridmet.filterDate(fire_date, fire_date.advance(1, "day"))
                                     .select("vpd").mean(), "vpd"),
        "vpd_7day":     zonal(window.select("vpd").mean(),  "vpd"),
        "tmmx_7day":    zonal(window.select("tmmx").mean(), "tmmx"),
        "pr_7day":      zonal(window.select("pr").sum(),    "pr"),
    })

fires = fires.map(add_fire_weather)

# ── Extract embeddings over polygons and export ───────────────────────────────
training = embedding_stack.reduceRegions(
    collection=fires, reducer=ee.Reducer.mean(),
    scale=30, crs="EPSG:4326", tileScale=8)

geemap.ee_to_csv(training, OUT_CSV)
print(f"Saved: {OUT_CSV}")

In [ ]:
# Verify columns
df = pd.read_csv("CO_2020_MTBS_20_AEF_GRIDMET_training_table.csv")
print("Shape:", df.shape)
print(df.columns.tolist())

## 4. Data Collection — Centroid Buffer Extraction

Uses 1 km centroid buffers instead of full polygons. Avoids geometry errors
on complex MTBS polygons and produces cleaner zonal statistics.

**Output:** `CO_2020_MTBS_20_AEF_GRIDMET_BUFFER_training_table.csv`

In [ ]:
import ee
import geemap
import pandas as pd

ee.Initialize()

# ── Parameters ────────────────────────────────────────────────────────────────
FIRE_YEAR   = 2020
BEFORE_YEAR = 2019
AFTER_YEAR  = 2021
N_FIRES     = 20
BUFFER_M    = 1000
FIRE_DATE_FIELD = "Ig_Date"
OUT_CSV = f"CO_{FIRE_YEAR}_MTBS_{N_FIRES}_AEF_GRIDMET_BUFFER_training_table.csv"

# ── Study area and MTBS ───────────────────────────────────────────────────────
states   = ee.FeatureCollection("TIGER/2018/States")
colorado = states.filter(ee.Filter.eq("NAME", "Colorado")).geometry()

def add_fire_year(f):
    d = ee.Date(f.get(FIRE_DATE_FIELD))
    return f.set({"fire_date": d.format("YYYY-MM-dd"),
                  "fire_year_from_date": d.get("year")})

fires_raw = (ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
               .map(add_fire_year)
               .filterBounds(colorado)
               .filter(ee.Filter.eq("fire_year_from_date", FIRE_YEAR))
               .limit(N_FIRES))

print("Selected fires:", fires_raw.size().getInfo())

# ── Convert to centroid buffers ───────────────────────────────────────────────
def make_centroid_buffer(f):
    fire_date    = ee.Date(f.get(FIRE_DATE_FIELD))
    lon          = ee.Number.parse(f.get("BurnBndLon"))
    lat          = ee.Number.parse(f.get("BurnBndLat"))
    burn_area_ha = ee.Number(f.get("BurnBndAc")).multiply(0.404686)  # acres → ha
    return (ee.Feature(ee.Geometry.Point([lon, lat]).buffer(BUFFER_M))
              .copyProperties(f)
              .set({"event_id":     ee.String("fire_").cat(ee.String(f.id())),
                    "fire_year":    FIRE_YEAR,
                    "fire_date":    fire_date.format("YYYY-MM-dd"),
                    "burn_area_ha": burn_area_ha,
                    "lon": lon, "lat": lat, "buffer_m": BUFFER_M}))

fires = fires_raw.map(make_centroid_buffer)

# ── AlphaEarth embeddings ─────────────────────────────────────────────────────
emb    = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
emb_b  = emb.filterDate(f"{BEFORE_YEAR}-01-01", f"{BEFORE_YEAR}-12-31").first()
emb_a  = emb.filterDate(f"{AFTER_YEAR}-01-01",  f"{AFTER_YEAR}-12-31").first()
bands  = emb_b.bandNames()

before = emb_b.select(bands, bands.map(lambda b: ee.String("before_").cat(b)))
after  = emb_a.select(bands, bands.map(lambda b: ee.String("after_").cat(b)))
delta  = emb_a.subtract(emb_b).select(bands, bands.map(lambda b: ee.String("delta_").cat(b)))
embedding_stack = before.addBands(after).addBands(delta)

print("Embedding bands:", embedding_stack.bandNames().size().getInfo())

# ── Quick extraction test ─────────────────────────────────────────────────────
test = embedding_stack.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=fires.first().geometry(),
    scale=100, maxPixels=1e9, bestEffort=True, tileScale=16).getInfo()

print("Test extraction keys:", len(test))
if not test:
    raise ValueError("Embedding extraction returned nothing — try a larger BUFFER_M.")

# ── Extract and export ────────────────────────────────────────────────────────
training = embedding_stack.reduceRegions(
    collection=fires, reducer=ee.Reducer.mean(), scale=100, tileScale=16)

geemap.ee_to_csv(training, OUT_CSV)
print(f"Saved: {OUT_CSV}")

df = pd.read_csv(OUT_CSV)
emb_cols = [c for c in df.columns if c.startswith(("before_","after_","delta_"))]
print("Shape:", df.shape, "| Embedding cols:", len(emb_cols))

## 5. Training Dataset — Multi-year Fire/Non-fire

Builds a balanced dataset across **2000–2023**. Each point gets climate and
embeddings computed from its own `fire_year`, so predictors are always
year-matched to the fire event.

- **Fire points (label=1):** centroid of each selected fire polygon  
- **Non-fire points (label=0):** ~2 km offset from fire centroid  

**Output:** `xgboost_fire_nonfire_climate_topo_embedding.csv`

In [ ]:
import ee
import pandas as pd
import os

ee.Initialize()

# ── Parameters ────────────────────────────────────────────────────────────────
SAFE_CRS   = "EPSG:4326"
SCALE      = 100
SEED       = 42
START_YEAR = 2000
END_YEAR   = 2023
N_FIRES    = 20
OUT_CSV    = "xgboost_fire_nonfire_climate_topo_embedding.csv"

# ── Study area ────────────────────────────────────────────────────────────────
states     = ee.FeatureCollection("TIGER/2018/States")
study_area = states.filter(ee.Filter.eq("NAME", "Colorado")).geometry()

# ── Load and clean MTBS ───────────────────────────────────────────────────────
def clean_fire(f):
    fire_year = ee.Date(f.get("Ig_Date")).get("year")
    geom = f.geometry().transform(SAFE_CRS, 1).buffer(0, 1).simplify(30)
    return ee.Feature(geom, {"fire_id":   f.get("Event_ID"),
                              "fire_name": f.get("Incid_Name"),
                              "fire_year": fire_year})

fires_clean = (ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
                 .filterBounds(study_area)
                 .map(clean_fire)
                 .filter(ee.Filter.gte("fire_year", START_YEAR))
                 .filter(ee.Filter.lte("fire_year", END_YEAR))
                 .randomColumn("rand", SEED).sort("rand").limit(N_FIRES))

print("Selected fires:", fires_clean.size().getInfo())

# ── Fire points (label = 1) ───────────────────────────────────────────────────
def make_fire_point(f):
    return ee.Feature(f.geometry().centroid(30),
                      {"fire_id": f.get("fire_id"), "fire_name": f.get("fire_name"),
                       "fire_year": f.get("fire_year"), "label": 1, "sample_type": "fire"})

# ── Non-fire points (label = 0) — deterministic ~2 km offset ─────────────────
def make_nonfire_point(f):
    coords = f.geometry().centroid(30).coordinates()
    lon, lat = ee.Number(coords.get(0)), ee.Number(coords.get(1))
    c1 = ee.Geometry.Point([lon.add(0.02),      lat.add(0.02)])
    c2 = ee.Geometry.Point([lon.subtract(0.02), lat.subtract(0.02)])
    geom = ee.Algorithms.If(study_area.contains(c1, 30), c1, c2)
    return ee.Feature(ee.Geometry(geom),
                      {"fire_id": f.get("fire_id"), "fire_name": f.get("fire_name"),
                       "fire_year": f.get("fire_year"), "label": 0, "sample_type": "non_fire"})

samples = fires_clean.map(make_fire_point).merge(fires_clean.map(make_nonfire_point))
print("Total sample points:", samples.size().getInfo())

# ── Topography (static) ───────────────────────────────────────────────────────
dem      = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("elevation")
topo_img = dem.addBands(ee.Terrain.slope(dem).rename("slope")) \
              .addBands(ee.Terrain.aspect(dem).rename("aspect"))

# ── TerraClimate annual summary helper ────────────────────────────────────────
terraclimate = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")

def climate_for_year(year, prefix):
    year  = ee.Number(year)
    start = ee.Date.fromYMD(year, 1, 1)
    col   = terraclimate.filterDate(start, start.advance(1, "year"))
    return (col.select("pr").sum().rename(prefix + "_ppt_sum")
               .addBands(col.select("tmmx").mean().multiply(0.1).rename(prefix + "_tmax_mean"))
               .addBands(col.select("tmmn").mean().multiply(0.1).rename(prefix + "_tmin_mean"))
               .addBands(col.select("vpd").mean().multiply(0.01).rename(prefix + "_vpd_mean")))

# ── AlphaEarth embedding helpers ─────────────────────────────────────────────
embedding_ic = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

def rename_embedding(img, prefix):
    names = img.bandNames()
    return img.rename(names.map(lambda b: ee.String(prefix).cat("_").cat(b)))

def raw_emb(year):
    year  = ee.Number(year)
    start = ee.Date.fromYMD(year, 1, 1)
    return (embedding_ic.filterDate(start, start.advance(1, "year"))
                        .filterBounds(study_area).mosaic().toFloat())

# ── Per-point sampling (year-aware) ──────────────────────────────────────────
def sample_one_point(f):
    yr       = ee.Number(f.get("fire_year"))
    pre_yr   = yr.subtract(1)
    post_yr  = yr.add(1)
    pre_raw  = raw_emb(pre_yr)
    post_raw = raw_emb(post_yr)
    img = (topo_img
           .addBands(climate_for_year(pre_yr,  "pre"))
           .addBands(climate_for_year(post_yr, "post"))
           .addBands(rename_embedding(pre_raw,  "pre_emb"))
           .addBands(rename_embedding(post_raw, "post_emb"))
           .addBands(rename_embedding(post_raw.subtract(pre_raw), "delta_emb"))
           .toFloat().unmask(-9999).setDefaultProjection(SAFE_CRS, None, SCALE))
    return img.sampleRegions(
        collection=ee.FeatureCollection([f]),
        properties=["fire_id","fire_name","fire_year","label","sample_type"],
        scale=SCALE, projection=SAFE_CRS, geometries=True, tileScale=16).first()

training = samples.map(sample_one_point)

# ── Save to CSV ───────────────────────────────────────────────────────────────
features = training.getInfo().get("features", [])
print("Features returned:", len(features))

rows = []
for feat in features:
    props = feat.get("properties", {}).copy()
    geom  = feat.get("geometry")
    if geom:
        props["longitude"] = geom["coordinates"][0]
        props["latitude"]  = geom["coordinates"][1]
    rows.append(props)

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print("Saved:", os.path.abspath(OUT_CSV))
print("Rows:", len(df))
if len(df):
    print(df["label"].value_counts())
    display(df.head())

## 6. Training Dataset — 2020 Fire/Non-fire (Clean Non-fire)

Fixed to **2020**. Non-fire points are placed randomly anywhere in Colorado
that is **outside all historical MTBS burn perimeters + 1 km safety buffer**,
preventing label contamination from previously burned areas.

- **Fire points (label=1):** centroids of selected 2020 burn polygons  
- **Non-fire points (label=0):** random, outside any-year MTBS + 1 km  

**Output:** `xgboost_fire_nonfire_2020_50_clean_nonfire.csv`

In [ ]:
import ee
import pandas as pd
import os

ee.Initialize()

# ── Parameters ────────────────────────────────────────────────────────────────
SAFE_CRS           = "EPSG:4326"
SCALE              = 100
SEED               = 42
FIRE_YEAR          = 2020
PRE_YEAR           = 2019
POST_YEAR          = 2021
N_FIRES            = 50
N_NONFIRE_TOTAL    = N_FIRES       # 1:1 class balance
NONFIRE_MIN_DIST_M = 1000
OUT_CSV            = "xgboost_fire_nonfire_2020_50_clean_nonfire.csv"

# ── Study area ────────────────────────────────────────────────────────────────
states     = ee.FeatureCollection("TIGER/2018/States")
study_area = states.filter(ee.Filter.eq("NAME", "Colorado")).geometry()

# ── Load and clean MTBS ───────────────────────────────────────────────────────
def clean_fire(f):
    fire_year = ee.Date(f.get("Ig_Date")).get("year")
    geom = f.geometry().transform(SAFE_CRS, 1).buffer(0, 1).simplify(30)
    return ee.Feature(geom, {"fire_id":   f.get("Event_ID"),
                              "fire_name": f.get("Incid_Name"),
                              "fire_year": fire_year})

mtbs_all   = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1").filterBounds(study_area)
mtbs_clean = mtbs_all.map(clean_fire)
print("MTBS in Colorado:", mtbs_all.size().getInfo())

fires_clean = (mtbs_clean.filter(ee.Filter.eq("fire_year", FIRE_YEAR))
                          .randomColumn("rand", SEED).sort("rand").limit(N_FIRES))
print("Selected 2020 fires:", fires_clean.size().getInfo())

# ── Fire points (label = 1) ───────────────────────────────────────────────────
def make_fire_point(f):
    return ee.Feature(f.geometry().centroid(30),
                      {"fire_id": f.get("fire_id"), "fire_name": f.get("fire_name"),
                       "fire_year": f.get("fire_year"), "label": 1, "sample_type": "fire"})

fire_points = fires_clean.map(make_fire_point)
print("Fire points:", fire_points.size().getInfo())

# ── Non-fire points — outside ALL historical MTBS burns + 1 km ───────────────
all_burn_geom       = mtbs_clean.geometry().buffer(0, 30)
burn_exclusion_geom = all_burn_geom.buffer(NONFIRE_MIN_DIST_M, 30)
nonfire_area        = study_area.difference(burn_exclusion_geom, 30).buffer(0, 30)

nonfire_raw = ee.FeatureCollection.randomPoints(
    region=nonfire_area, points=N_NONFIRE_TOTAL, seed=SEED, maxError=30)

nonfire_points = nonfire_raw.map(
    lambda f: ee.Feature(f.geometry(),
                         {"fire_id": "non_fire", "fire_name": "non_fire",
                          "fire_year": FIRE_YEAR, "label": 0, "sample_type": "non_fire"}))
print("Non-fire points:", nonfire_points.size().getInfo())

# ── Validate non-fire placement ───────────────────────────────────────────────
def check_nonfire(f):
    return f.set({"inside_burn_polygon": all_burn_geom.contains(f.geometry(), 30),
                  "distance_to_burn_m":  f.geometry().distance(all_burn_geom, 30)})

nonfire_check = nonfire_points.map(check_nonfire)
print("Non-fire inside burn:",
      nonfire_check.filter(ee.Filter.eq("inside_burn_polygon", True)).size().getInfo())

samples = fire_points.merge(nonfire_points)
print("Total samples:", samples.size().getInfo())

# ── Topography ────────────────────────────────────────────────────────────────
dem      = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("elevation")
topo_img = dem.addBands(ee.Terrain.slope(dem).rename("slope")) \
              .addBands(ee.Terrain.aspect(dem).rename("aspect"))

# ── TerraClimate ──────────────────────────────────────────────────────────────
terraclimate = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")

def climate_for_year(year, prefix):
    start = ee.Date.fromYMD(year, 1, 1)
    col   = terraclimate.filterDate(start, start.advance(1, "year"))
    return (col.select("pr").sum().rename(prefix + "_ppt_sum")
               .addBands(col.select("tmmx").mean().multiply(0.1).rename(prefix + "_tmax_mean"))
               .addBands(col.select("tmmn").mean().multiply(0.1).rename(prefix + "_tmin_mean"))
               .addBands(col.select("vpd").mean().multiply(0.01).rename(prefix + "_vpd_mean")))

climate_pre  = climate_for_year(PRE_YEAR,  "pre")
climate_post = climate_for_year(POST_YEAR, "post")

# ── AlphaEarth embeddings ─────────────────────────────────────────────────────
embedding_ic = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

def raw_emb(year):
    start = ee.Date.fromYMD(year, 1, 1)
    return (embedding_ic.filterDate(start, start.advance(1, "year"))
                        .filterBounds(study_area).mosaic().toFloat().unmask(-9999))

def rename_embedding(img, prefix):
    names = img.bandNames()
    return img.rename(names.map(lambda b: ee.String(prefix).cat("_").cat(b)))

emb_pre_raw  = raw_emb(PRE_YEAR)
emb_post_raw = raw_emb(POST_YEAR)

# ── Combine predictors and sample ────────────────────────────────────────────
predictor_img = (topo_img
                 .addBands(climate_pre)
                 .addBands(climate_post)
                 .addBands(rename_embedding(emb_pre_raw,                          "pre_emb"))
                 .addBands(rename_embedding(emb_post_raw,                         "post_emb"))
                 .addBands(rename_embedding(emb_post_raw.subtract(emb_pre_raw),   "delta_emb"))
                 .toFloat().unmask(-9999).setDefaultProjection(SAFE_CRS, None, SCALE))

training = predictor_img.sampleRegions(
    collection=samples,
    properties=["fire_id","fire_name","fire_year","label","sample_type"],
    scale=SCALE, projection=SAFE_CRS, geometries=True, tileScale=16)

# ── Save ──────────────────────────────────────────────────────────────────────
features = training.getInfo().get("features", [])
print("Features returned:", len(features))

rows = []
for feat in features:
    props = feat.get("properties", {}).copy()
    geom  = feat.get("geometry")
    if geom:
        props["longitude"] = geom["coordinates"][0]
        props["latitude"]  = geom["coordinates"][1]
    rows.append(props)

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print("Saved:", os.path.abspath(OUT_CSV))
print("Rows:", len(df))
if len(df):
    print(df["label"].value_counts())
    display(df.head())

## 7. Export Shapefiles

Export 2020 MTBS fire perimeters and validated non-fire points as
shapefiles for use in GIS software.

In [ ]:
import ee
import geemap
import os

ee.Initialize()

SAFE_CRS           = "EPSG:4326"
FIRE_YEAR          = 2020
N_NONFIRE_POINTS   = 50
NONFIRE_MIN_DIST_M = 1000
SEED               = 42
OUT_DIR            = "mtbs_2020_fire_nonfire_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Study area and MTBS ───────────────────────────────────────────────────────
states     = ee.FeatureCollection("TIGER/2018/States")
study_area = states.filter(ee.Filter.eq("NAME", "Colorado")).geometry()

def clean_fire(f):
    fire_year = ee.Date(f.get("Ig_Date")).get("year")
    geom = f.geometry().transform(SAFE_CRS, 1).buffer(0, 1).simplify(30)
    return ee.Feature(geom, {"fire_id":      f.get("Event_ID"),
                              "fire_name":    f.get("Incid_Name"),
                              "fire_year":    fire_year,
                              "ig_date":      f.get("Ig_Date"),
                              "label":        1,
                              "sample_type":  "fire_perimeter"})

mtbs_clean = (ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
                .filterBounds(study_area).map(clean_fire))

mtbs_2020  = mtbs_clean.filter(ee.Filter.eq("fire_year", FIRE_YEAR))
print("2020 MTBS fires:", mtbs_2020.size().getInfo())

# ── Non-fire points outside any-year MTBS + 1 km ─────────────────────────────
all_burn_geom   = mtbs_clean.geometry().buffer(0, 30)
nonfire_area    = study_area.difference(all_burn_geom.buffer(NONFIRE_MIN_DIST_M, 30), 30).buffer(0, 30)

nonfire_raw = ee.FeatureCollection.randomPoints(
    region=nonfire_area, points=N_NONFIRE_POINTS, seed=SEED, maxError=30)

def check_and_label(f):
    return f.set({"fire_id":   "non_fire", "fire_name":  "non_fire",
                  "fire_year": FIRE_YEAR,  "label":      0,
                  "sample_type": "non_fire",
                  "rule": "outside_any_year_MTBS_plus_1000m",
                  "inside_any_mtbs_burn":             all_burn_geom.contains(f.geometry(), 30),
                  "distance_to_nearest_mtbs_burn_m":  f.geometry().distance(all_burn_geom, 30)})

nonfire_points = nonfire_raw.map(check_and_label)
print("Non-fire points inside any burn:",
      nonfire_points.filter(ee.Filter.eq("inside_any_mtbs_burn", True)).size().getInfo())

# ── Export ────────────────────────────────────────────────────────────────────
fire_shp    = os.path.join(OUT_DIR, "mtbs_2020_fire_perimeters.shp")
nonfire_shp = os.path.join(OUT_DIR, "nonfire_points_outside_any_year_mtbs.shp")

geemap.ee_export_vector(mtbs_2020,       filename=fire_shp)
geemap.ee_export_vector(nonfire_points,  filename=nonfire_shp)

print("Saved:", os.path.abspath(fire_shp))
print("Saved:", os.path.abspath(nonfire_shp))

## 8. Final Pre-fire Training Dataset (Recommended)

**This is the primary dataset for modelling.** Uses only **pre-fire (2019)
embeddings and climate** to avoid data leakage. Fire points are random
points within each 2020 burn polygon (not just centroids).

- **Fire (label=1):** 20 random points per polygon × up to 50 fires  
- **Non-fire (label=0):** 200 random points outside all-year MTBS + 1 km  

**Outputs:** CSV · GeoPackage · Shapefile  
**Output directory:** `mtbs_2020_prefire_20firepts_200nonfire/`

In [ ]:
import ee
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
import os

ee.Initialize()

# ── Parameters ────────────────────────────────────────────────────────────────
SAFE_CRS               = "EPSG:4326"
SCALE                  = 100
SEED                   = 42
FIRE_YEAR              = 2020
PRE_YEAR               = 2019
N_FIRES                = 50
N_FIRE_POINTS_PER_FIRE = 20
N_NONFIRE_POINTS       = 200
NONFIRE_MIN_DIST_M     = 1000

OUT_DIR = "mtbs_2020_prefire_20firepts_200nonfire"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV       = os.path.join(OUT_DIR, "fire_nonfire_2020_prefire_predictors.csv")
OUT_GPKG      = os.path.join(OUT_DIR, "fire_nonfire_2020_prefire_predictors.gpkg")
OUT_SHP       = os.path.join(OUT_DIR, "fire_nonfire_2020_prefire_predictors.shp")
OUT_GPKG_FIRE = os.path.join(OUT_DIR, "mtbs_2020_fire_perimeters.gpkg")
OUT_SHP_FIRE  = os.path.join(OUT_DIR, "mtbs_2020_fire_perimeters.shp")

# ── Study area and MTBS ───────────────────────────────────────────────────────
states     = ee.FeatureCollection("TIGER/2018/States")
study_area = states.filter(ee.Filter.eq("NAME", "Colorado")).geometry()

def clean_fire(f):
    fire_year = ee.Date(f.get("Ig_Date")).get("year")
    geom = f.geometry().transform(SAFE_CRS, 1).buffer(0, 1).simplify(60)
    return ee.Feature(geom, {"fire_id":   f.get("Event_ID"),
                              "fire_name": f.get("Incid_Name"),
                              "fire_year": fire_year,
                              "ig_date":   f.get("Ig_Date")})

mtbs_clean = (ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
                .filterBounds(study_area).map(clean_fire))
print("All MTBS fires in Colorado:", mtbs_clean.size().getInfo())

fires_2020 = (mtbs_clean.filter(ee.Filter.eq("fire_year", FIRE_YEAR))
                         .randomColumn("rand", SEED).sort("rand").limit(N_FIRES))
print("Selected 2020 fires:", fires_2020.size().getInfo())

# ── Fire points — random points within each burn polygon ─────────────────────
def make_fire_points(f):
    return (ee.FeatureCollection.randomPoints(
                region=f.geometry(), points=N_FIRE_POINTS_PER_FIRE, seed=SEED, maxError=30)
              .map(lambda pt: ee.Feature(pt.geometry(),
                                         {"fire_id": f.get("fire_id"),
                                          "fire_name": f.get("fire_name"),
                                          "fire_year": f.get("fire_year"),
                                          "label": 1, "sample_type": "fire"})))

fire_points = fires_2020.map(make_fire_points).flatten()
print("Fire points:", fire_points.size().getInfo())

# ── Non-fire points — outside ALL MTBS burns + 1 km ─────────────────────────
all_burn_geom = mtbs_clean.geometry().buffer(0, 30)
nonfire_area  = study_area.difference(all_burn_geom.buffer(NONFIRE_MIN_DIST_M, 30), 30).buffer(0, 30)

nonfire_raw   = ee.FeatureCollection.randomPoints(
    region=nonfire_area, points=N_NONFIRE_POINTS, seed=SEED, maxError=30)

def label_nonfire(f):
    return f.set({"fire_id":     "non_fire", "fire_name":  "non_fire",
                  "fire_year":   FIRE_YEAR,  "label":      0, "sample_type": "non_fire",
                  "inside_any_mtbs_burn":            all_burn_geom.contains(f.geometry(), 30),
                  "distance_to_nearest_mtbs_burn_m": f.geometry().distance(all_burn_geom, 30)})

nonfire_points = nonfire_raw.map(label_nonfire)
print("Non-fire points:", nonfire_points.size().getInfo())
print("Non-fire inside burn:",
      nonfire_points.filter(ee.Filter.eq("inside_any_mtbs_burn", True)).size().getInfo())

samples = fire_points.merge(nonfire_points)
print("Total samples:", samples.size().getInfo())

# ── Topography ────────────────────────────────────────────────────────────────
dem      = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("elevation")
topo_img = dem.addBands(ee.Terrain.slope(dem).rename("slope")) \
              .addBands(ee.Terrain.aspect(dem).rename("aspect"))

# ── Pre-burn TerraClimate (2019) ──────────────────────────────────────────────
terraclimate = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")

def climate_for_year(year, prefix):
    start = ee.Date.fromYMD(year, 1, 1)
    col   = terraclimate.filterDate(start, start.advance(1, "year"))
    return (col.select("pr").sum().rename(prefix + "_ppt_sum")
               .addBands(col.select("tmmx").mean().multiply(0.1).rename(prefix + "_tmax_mean"))
               .addBands(col.select("tmmn").mean().multiply(0.1).rename(prefix + "_tmin_mean"))
               .addBands(col.select("vpd").mean().multiply(0.01).rename(prefix + "_vpd_mean")))

climate_pre = climate_for_year(PRE_YEAR, "pre")

# ── Pre-burn AlphaEarth embedding (2019) ─────────────────────────────────────
embedding_ic = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

def raw_emb(year):
    start = ee.Date.fromYMD(year, 1, 1)
    return (embedding_ic.filterDate(start, start.advance(1, "year"))
                        .filterBounds(study_area).mosaic().toFloat().unmask(-9999))

def rename_embedding(img, prefix):
    names = img.bandNames()
    return img.rename(names.map(lambda b: ee.String(prefix).cat("_").cat(b)))

emb_pre = rename_embedding(raw_emb(PRE_YEAR), "pre_emb")

# ── Combine predictors and sample ─────────────────────────────────────────────
predictor_img = (topo_img.addBands(climate_pre).addBands(emb_pre)
                         .toFloat().unmask(-9999).setDefaultProjection(SAFE_CRS, None, SCALE))

training = predictor_img.sampleRegions(
    collection=samples,
    properties=["fire_id","fire_name","fire_year","label","sample_type",
                "inside_any_mtbs_burn","distance_to_nearest_mtbs_burn_m"],
    scale=SCALE, projection=SAFE_CRS, geometries=True, tileScale=16)
print("Sampling complete.")

# ── Helper: EE FeatureCollection → GeoDataFrame ──────────────────────────────
def ee_fc_to_gdf(fc):
    rows = []
    for feat in fc.getInfo()["features"]:
        props = feat.get("properties", {}).copy()
        geom  = feat.get("geometry")
        if geom:
            props["geometry"] = shape(geom)
        rows.append(props)
    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")

# ── Save point outputs ────────────────────────────────────────────────────────
gdf_points = ee_fc_to_gdf(training)
gdf_points["longitude"] = gdf_points.geometry.x
gdf_points["latitude"]  = gdf_points.geometry.y

pd.DataFrame(gdf_points.drop(columns="geometry")).to_csv(OUT_CSV, index=False)
gdf_points.to_file(OUT_GPKG, driver="GPKG")
gdf_points.to_file(OUT_SHP)

print("Rows saved:", len(gdf_points))
print("Label counts:\n", gdf_points["label"].value_counts())
print("Pre-embedding cols:", len([c for c in gdf_points.columns if c.startswith("pre_emb_")]))

# ── Save fire perimeters ──────────────────────────────────────────────────────
gdf_fire = ee_fc_to_gdf(fires_2020)
gdf_fire.to_file(OUT_GPKG_FIRE, driver="GPKG")
gdf_fire.to_file(OUT_SHP_FIRE)
print("Fire perimeters saved:", os.path.abspath(OUT_GPKG_FIRE))

## 9. XGBoost Classifier

Binary fire / non-fire classifier using **pre-fire only** predictors
(embeddings + climate + topography). Stratified 80/20 train/test split.

**Input:** `xgboost_fire_nonfire_2020_50_clean_nonfire.csv`

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                              classification_report, precision_score, recall_score, f1_score)

os.makedirs("figures", exist_ok=True)

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv("xgboost_fire_nonfire_2020_50_clean_nonfire.csv").replace(-9999, np.nan)
print(f"Rows: {len(df)} | Columns: {len(df.columns)}")
print(df["label"].value_counts())

# ── Features — pre-fire only (no data leakage) ────────────────────────────────
pre_embedding_cols = [c for c in df.columns if c.startswith("pre_emb_")]
pre_climate_cols   = [c for c in ["pre_ppt_sum","pre_tmax_mean","pre_tmin_mean","pre_vpd_mean"]
                      if c in df.columns]
static_cols        = [c for c in ["elevation","slope","aspect","longitude","latitude"]
                      if c in df.columns]
feature_cols       = pre_embedding_cols + pre_climate_cols + static_cols

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["label"].astype(int)
print(f"Features — embedding: {len(pre_embedding_cols)} | climate: {len(pre_climate_cols)} | static: {len(static_cols)}")

# ── Stratified 80/20 split ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.20, random_state=42, stratify=y)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# ── Fit XGBoost ───────────────────────────────────────────────────────────────
clf = XGBClassifier(n_estimators=150, learning_rate=0.04, max_depth=2,
                    min_child_weight=1, subsample=0.85, colsample_bytree=0.85,
                    eval_metric="logloss", random_state=42)
clf.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────
train_prob = clf.predict_proba(X_train)[:, 1]
test_prob  = clf.predict_proba(X_test)[:, 1]
train_pred = (train_prob >= 0.5).astype(int)
test_pred  = (test_prob  >= 0.5).astype(int)

print(f"\nTrain acc: {accuracy_score(y_train, train_pred):.3f} | AUC: {roc_auc_score(y_train, train_prob):.3f}")
print(f"Test  acc: {accuracy_score(y_test,  test_pred):.3f} | AUC: {roc_auc_score(y_test,  test_prob):.3f}")
print(f"Precision: {precision_score(y_test, test_pred):.3f} | Recall: {recall_score(y_test, test_pred):.3f} | F1: {f1_score(y_test, test_pred):.3f}")
print("\n", classification_report(y_test, test_pred, target_names=["non_fire","fire"]))

# ── Save predictions ──────────────────────────────────────────────────────────
df_train = df_train.copy(); df_train["set"] = "train"
df_test  = df_test.copy();  df_test["set"]  = "test"
for split_df, prob, pred in [(df_train, train_prob, train_pred), (df_test, test_prob, test_pred)]:
    split_df["pred_fire_probability"] = prob
    split_df["pred_label"]            = pred
df_pred = pd.concat([df_train, df_test])

# ── Feature importance ────────────────────────────────────────────────────────
importance = (pd.DataFrame({"feature": feature_cols, "importance": clf.feature_importances_})
                .sort_values("importance", ascending=False))
print("\nTop 20 predictors:\n", importance.head(20).to_string())

# ── Visualizations ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Confusion matrix
cm = confusion_matrix(y_test, test_pred)
axes[0,0].imshow(cm); axes[0,0].set_title("Confusion matrix (test)")
axes[0,0].set_xlabel("Predicted"); axes[0,0].set_ylabel("Observed")
axes[0,0].set_xticks([0,1]); axes[0,0].set_xticklabels(["Non-fire","Fire"])
axes[0,0].set_yticks([0,1]); axes[0,0].set_yticklabels(["Non-fire","Fire"])
for i in range(2):
    for j in range(2):
        axes[0,0].text(j, i, cm[i,j], ha="center", va="center", color="white")

# Histogram
axes[0,1].hist(df_test.loc[df_test["label"]==0,"pred_fire_probability"], alpha=0.6, label="Non-fire")
axes[0,1].hist(df_test.loc[df_test["label"]==1,"pred_fire_probability"], alpha=0.6, label="Fire")
axes[0,1].set_xlabel("Predicted fire probability"); axes[0,1].set_ylabel("Count")
axes[0,1].set_title("Predicted probability distribution (test)"); axes[0,1].legend()

# Top 20 feature importance
top20 = importance.head(20).sort_values("importance")
axes[1,0].barh(top20["feature"], top20["importance"])
axes[1,0].set_xlabel("Feature importance"); axes[1,0].set_title("Top 20 predictors")

# Spatial scatter
sc = axes[1,1].scatter(df_test["longitude"], df_test["latitude"],
                        c=df_test["pred_fire_probability"], s=60)
for _, row in df_test.iterrows():
    axes[1,1].text(row["longitude"], row["latitude"], row["sample_type"], fontsize=7)
plt.colorbar(sc, ax=axes[1,1], label="Predicted fire probability")
axes[1,1].set_xlabel("Longitude"); axes[1,1].set_ylabel("Latitude")
axes[1,1].set_title("Test points: predicted fire probability")

plt.suptitle("XGBoost Classifier — Pre-fire Predictors Only", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/xgb_classifier_diagnostics.png", dpi=150)
plt.show()

# ── Save outputs ──────────────────────────────────────────────────────────────
joblib.dump(clf,          "xgb_fire_probability_prefire_model.pkl")
joblib.dump(feature_cols, "feature_cols_prefire.pkl")
importance.to_csv("xgb_feature_importance_prefire.csv", index=False)
df_pred.to_csv("fire_nonfire_train_test_predictions_prefire.csv", index=False)
print("\nSaved: xgb_fire_probability_prefire_model.pkl, feature_cols_prefire.pkl")

## 10. Random Forest Classifier

Same pre-fire feature set and 80/20 stratified split as Section 9,
using a Random Forest classifier for comparison.

**Input:** `xgboost_fire_nonfire_2020_50_clean_nonfire.csv`

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                              classification_report, precision_score, recall_score, f1_score)

os.makedirs("figures", exist_ok=True)

# ── Load data and define features (same as Section 9) ────────────────────────
df = pd.read_csv("xgboost_fire_nonfire_2020_50_clean_nonfire.csv").replace(-9999, np.nan)

pre_embedding_cols = [c for c in df.columns if c.startswith("pre_emb_")]
pre_climate_cols   = [c for c in ["pre_ppt_sum","pre_tmax_mean","pre_tmin_mean","pre_vpd_mean"]
                      if c in df.columns]
static_cols        = [c for c in ["elevation","slope","aspect","longitude","latitude"]
                      if c in df.columns]
feature_cols       = pre_embedding_cols + pre_climate_cols + static_cols

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["label"].astype(int)

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.20, random_state=42, stratify=y)

# ── Fit Random Forest ─────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=500, max_depth=6, min_samples_leaf=2,
                             max_features="sqrt", bootstrap=True, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────
train_prob = rf.predict_proba(X_train)[:, 1]
test_prob  = rf.predict_proba(X_test)[:, 1]
train_pred = rf.predict(X_train)
test_pred  = rf.predict(X_test)

print(f"Train acc: {accuracy_score(y_train, train_pred):.3f} | AUC: {roc_auc_score(y_train, train_prob):.3f}")
print(f"Test  acc: {accuracy_score(y_test,  test_pred):.3f} | AUC: {roc_auc_score(y_test,  test_prob):.3f}")
print(f"Precision: {precision_score(y_test, test_pred):.3f} | Recall: {recall_score(y_test, test_pred):.3f} | F1: {f1_score(y_test, test_pred):.3f}")
print("\n", classification_report(y_test, test_pred, target_names=["non_fire","fire"]))

# ── Save predictions ──────────────────────────────────────────────────────────
df_train = df_train.copy(); df_train["set"] = "train"
df_test  = df_test.copy();  df_test["set"]  = "test"
for split_df, prob, pred in [(df_train, train_prob, train_pred), (df_test, test_prob, test_pred)]:
    split_df["pred_fire_probability"] = prob
    split_df["pred_label"]            = pred
df_pred = pd.concat([df_train, df_test])

# ── Feature importance ────────────────────────────────────────────────────────
importance = (pd.DataFrame({"feature": feature_cols, "importance": rf.feature_importances_})
                .sort_values("importance", ascending=False))
print("\nTop 20 predictors:\n", importance.head(20).to_string())

# ── Visualizations ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

cm = confusion_matrix(y_test, test_pred)
axes[0,0].imshow(cm); axes[0,0].set_title("Confusion matrix (test)")
axes[0,0].set_xlabel("Predicted"); axes[0,0].set_ylabel("Observed")
axes[0,0].set_xticks([0,1]); axes[0,0].set_xticklabels(["Non-fire","Fire"])
axes[0,0].set_yticks([0,1]); axes[0,0].set_yticklabels(["Non-fire","Fire"])
for i in range(2):
    for j in range(2):
        axes[0,0].text(j, i, cm[i,j], ha="center", va="center", color="white")

axes[0,1].hist(df_test.loc[df_test["label"]==0,"pred_fire_probability"], alpha=0.6, label="Non-fire")
axes[0,1].hist(df_test.loc[df_test["label"]==1,"pred_fire_probability"], alpha=0.6, label="Fire")
axes[0,1].set_title("Predicted probability distribution (test)")
axes[0,1].set_xlabel("Predicted fire probability"); axes[0,1].legend()

top20 = importance.head(20).sort_values("importance")
axes[1,0].barh(top20["feature"], top20["importance"])
axes[1,0].set_xlabel("Feature importance"); axes[1,0].set_title("Top 20 predictors")

sc = axes[1,1].scatter(df_test["longitude"], df_test["latitude"],
                        c=df_test["pred_fire_probability"], s=60)
for _, row in df_test.iterrows():
    axes[1,1].text(row["longitude"], row["latitude"],
                   "F" if row["sample_type"] == "fire" else "NF", fontsize=7)
plt.colorbar(sc, ax=axes[1,1], label="Predicted fire probability")
axes[1,1].set_xlabel("Longitude"); axes[1,1].set_ylabel("Latitude")
axes[1,1].set_title("Test points: predicted fire probability")

plt.suptitle("Random Forest Classifier — Pre-fire Predictors Only", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/rf_classifier_diagnostics.png", dpi=150)
plt.show()

# ── Save outputs ──────────────────────────────────────────────────────────────
joblib.dump(rf,           "rf_fire_probability_prefire_model.pkl")
joblib.dump(feature_cols, "rf_feature_cols_prefire.pkl")
importance.to_csv("rf_feature_importance_prefire.csv", index=False)
df_pred.to_csv("rf_fire_nonfire_train_test_predictions.csv", index=False)
print("\nSaved: rf_fire_probability_prefire_model.pkl, rf_feature_cols_prefire.pkl")

## 11. XGBoost Area Regression (LOO-CV)

Predicts **burned area (ha)** using leave-one-out cross-validation on the
20-fire polygon extraction dataset. Area is log-transformed before fitting
to handle the heavily right-skewed fire size distribution.

**Input:** `CO_2020_MTBS_20_AEF_GRIDMET_training_table.csv`

In [ ]:
import pandas as pd
import numpy as np
import joblib

from xgboost import XGBRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv("CO_2020_MTBS_20_AEF_GRIDMET_training_table.csv")
df = df.dropna(subset=["burn_area_ha"]).copy()
df["log_area"] = np.log1p(df["burn_area_ha"])
print(f"Fires: {len(df)} | Area range: {df['burn_area_ha'].min():.0f}–{df['burn_area_ha'].max():.0f} ha")

# ── Features: AEF embeddings + GRIDMET fire-weather ──────────────────────────
feature_cols  = [c for c in df.columns if c.startswith(("before_","after_","delta_"))]
feature_cols += [c for c in ["vpd_fire_day","vpd_7day","tmmx_7day","pr_7day"] if c in df.columns]

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["log_area"]
print(f"Features: {len(feature_cols)}")

# ── LOO-CV ────────────────────────────────────────────────────────────────────
model = XGBRegressor(n_estimators=120, learning_rate=0.05, max_depth=2,
                     subsample=0.85, colsample_bytree=0.85,
                     objective="reg:squarederror", random_state=42)

pred_log  = cross_val_predict(model, X, y, cv=LeaveOneOut())
pred_area = np.expm1(pred_log)

print("\nLOO-CV performance")
print(f"  MAE  : {mean_absolute_error(df['burn_area_ha'], pred_area):,.0f} ha")
print(f"  RMSE : {mean_squared_error(df['burn_area_ha'], pred_area, squared=False):,.0f} ha")
print(f"  R²   : {r2_score(df['burn_area_ha'], pred_area):.3f}")

df["cv_pred_area_ha"] = pred_area

# ── Fit final model on all data ───────────────────────────────────────────────
model.fit(X, y)

joblib.dump(model,        "xgb_fire_area_toy.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")
df.to_csv("training_with_predictions.csv", index=False)
print("\nSaved: xgb_fire_area_toy.pkl, training_with_predictions.csv")

## 12. Streamlit Digital Twin App

Interactive web app that lets users:
1. **Select an existing demo fire** from the training set and adjust VPD to explore scenario predictions
2. **Enter any lon/lat** in Colorado and get a real-time burned area prediction by querying
   the most recent AlphaEarth annual embedding from Google Earth Engine

Run with: `streamlit run app.py`  
**Requires:** `xgb_fire_area_toy.pkl`, `feature_cols.pkl`, `training_with_predictions.csv`

In [ ]:
import ee
import joblib
import numpy as np
import pandas as pd
import streamlit as st

ee.Initialize()

# ── Page config ───────────────────────────────────────────────────────────────
st.set_page_config(page_title="Embedding-informed Fire Digital Twin", layout="wide")
st.title("Embedding-informed Fire Digital Twin")

# ── Load model and data ───────────────────────────────────────────────────────
model        = joblib.load("xgb_fire_area_toy.pkl")
feature_cols = joblib.load("feature_cols.pkl")
training_df  = pd.read_csv("training_with_predictions.csv")

emb                   = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
LATEST_EMBEDDING_YEAR = 2024   # update when a newer annual layer is released
BASELINE_YEAR         = 2019

# ── Helpers ───────────────────────────────────────────────────────────────────
def get_embedding(lon, lat, buffer_m, year, prefix):
    region = ee.Geometry.Point([lon, lat]).buffer(buffer_m)
    vals   = (emb.filterDate(f"{year}-01-01", f"{year}-12-31").first()
                 .reduceRegion(reducer=ee.Reducer.mean(), geometry=region,
                               scale=30, maxPixels=1e9, bestEffort=True).getInfo())
    return {f"{prefix}_{k}": v for k, v in vals.items()}


def build_row(lon, lat, buffer_m, user_vpd):
    before  = get_embedding(lon, lat, buffer_m, BASELINE_YEAR,         "before")
    current = get_embedding(lon, lat, buffer_m, LATEST_EMBEDDING_YEAR, "after")
    row = {**before, **current}
    for bk in [k for k in row if k.startswith("before_")]:
        ak = bk.replace("before_", "after_")
        if ak in row:
            row[bk.replace("before_", "delta_")] = row[ak] - row[bk]
    row.update({"vpd_fire_day": user_vpd, "vpd_7day": user_vpd, "tmmx_7day": 300, "pr_7day": 0})
    out = pd.DataFrame([row])
    for col in feature_cols:
        if col not in out.columns:
            out[col] = 0
    return out[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)


def predict_area(X):
    return float(np.expm1(model.predict(X)[0]))

# ── Sidebar controls ──────────────────────────────────────────────────────────
mode     = st.sidebar.radio("Prediction mode", ["Existing demo fire", "New user location"])
user_vpd = st.sidebar.slider("Scenario VPD (kPa)", min_value=0.0, max_value=8.0, value=2.5, step=0.1)

# ── Mode 1: existing demo fire ────────────────────────────────────────────────
if mode == "Existing demo fire":
    idx = st.sidebar.selectbox("Select demo fire", training_df.index.tolist())
    row = training_df.loc[[idx]].copy()
    for col in ["vpd_7day", "vpd_fire_day"]:
        if col in row.columns:
            row[col] = user_vpd
    X         = row[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    pred_area = predict_area(X)
    observed  = row["burn_area_ha"].iloc[0]
    c1, c2 = st.columns(2)
    c1.metric("Observed burned area",        f"{observed:,.1f} ha")
    c2.metric("Scenario predicted area",     f"{pred_area:,.1f} ha")
    st.dataframe(row[["burn_area_ha","vpd_7day","vpd_fire_day"]])

# ── Mode 2: new user location ─────────────────────────────────────────────────
else:
    lon      = st.sidebar.number_input("Longitude", value=-105.25, format="%.6f")
    lat      = st.sidebar.number_input("Latitude",  value=39.50,   format="%.6f")
    buffer_m = st.sidebar.slider("Buffer radius (m)", min_value=250, max_value=5000,
                                  value=1000, step=250)
    if st.sidebar.button("Run digital twin"):
        with st.spinner("Querying GEE embedding and running model..."):
            X_new     = build_row(lon, lat, buffer_m, user_vpd)
            pred_area = predict_area(X_new)
        c1, c2, c3 = st.columns(3)
        c1.metric("Predicted affected area", f"{pred_area:,.1f} ha")
        c2.metric("Scenario VPD",            f"{user_vpd:.2f} kPa")
        c3.metric("Buffer radius",           f"{buffer_m:,} m")
        st.caption("Uses most recent annual AlphaEarth embedding as current-condition proxy.")
        st.dataframe(X_new)
        st.download_button("Download scenario inputs / outputs",
                           X_new.assign(predicted_area_ha=pred_area).to_csv(index=False),
                           "digital_twin_prediction.csv", "text/csv")
    else:
        st.info("Set lon/lat and VPD in the sidebar, then click Run digital twin.")